In [4]:
# CS-340 Project Two: Grazioso Salvare Rescue Animal Dashboard
from jupyter_dash import JupyterDash

# Dashboard, visualization, and data dependencies
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Supporting libraries
import os

# Data processing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Data access and MongoDB connection

username = "aacuser"
password = "JayJay81"

# Authenticate through the AnimalShelter CRUD module
db = AnimalShelter(username, password)

# Load animal records from MongoDB
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectIds, which cannot be serialized in the Dash table.
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################

#Dashboard Layout
app = JupyterDash(__name__)

# Load the Grazioso Salvare’s logo.
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
   
    html.Div([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'width': '220px', 'height': 'auto'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H1('Grazioso Salvare Rescue Animal Dashboard'),
        html.H4('Developed by Jeremiah (Jay) Case | CS-340')
    ], style={'textAlign': 'center'}),

        
# Rescue-type filters for the interactive data table.
dcc.RadioItems(
    id='filter-type',
    options=[
        {'label': 'Water Rescue', 'value': 'Water'},
        {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
        {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
        {'label': 'Reset', 'value': 'Reset'}
    ],
    value='Reset',
    labelStyle={
        'display': 'inline-block',
        'marginRight': '20px'
    },
    style={
        'textAlign': 'center',
        'padding': '15px'
    }
),
    html.Hr(),
    
dash_table.DataTable(
    id='datatable-id',
    columns=[
        {'name': i, 'id': i, 'deletable': False, 'selectable': True}
        for i in df.columns
    ],
    data=df.to_dict('records'),
    page_size=10,
    page_action='native',
    sort_action='native',
    filter_action='native',
    row_selectable='single',
    selected_rows=[0],
    style_table={'overflowX': 'auto'},
    style_cell={
        'textAlign': 'left',
        'padding': '8px',
        'minWidth': '120px',
        'maxWidth': '250px',
        'overflow': 'hidden',
        'textOverflow': 'ellipsis'
    },
    style_header={
        'backgroundColor': '#F0F0F0',
        'fontWeight': 'bold'
    }
),

   
    html.Br(),
    html.Hr(),
# Display the breed chart and geolocation map side-by-side
   
html.Div(
    style={
        'display': 'flex',
        'flexWrap': 'wrap',
        'width': '100%',
        'gap': '20px',
        'alignItems': 'flex-start'
    },
    children=[
        html.Div(
            id='graph-id',
            style={
                'flex': '1 1 450px',
                'minWidth': '350px'
            }
        ),
        html.Div(
            id='map-id',
            style={
                'flex': '1 1 450px',
                'minWidth': '350px'
            }
        )
    ])
])
    


#############################################
# Interaction Between Components / Controller
#############################################


# Dashboard callbacks
    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type == 'Water':
        breeds = ['Labrador Retriever Mix',
                  'Chesapeake Bay Retriever', 'Newfoundland']
        sex = 'Intact Female'
        min_age, max_age = 26, 156

    elif filter_type == 'Mountain':
        breeds = ['German Shepherd', 'Alaskan Malamute',
                  'Old English Sheepdog', 'Siberian Husky',
                  'Rottweiler']
        sex = 'Intact Male'
        min_age, max_age = 26, 156

    elif filter_type == 'Disaster':
        breeds = ['Doberman Pinscher', 'German Shepherd',
                  'Golden Retriever', 'Bloodhound', 'Rottweiler']
        sex = 'Intact Male'
        min_age, max_age = 20, 300

    else:
        query = {}

    if filter_type != 'Reset':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': breeds},
            'sex_upon_outcome': sex,
            'age_upon_outcome_in_weeks': {
                '$gte': min_age, '$lte': max_age
            }
        }

    filtered_df = pd.DataFrame.from_records(db.read(query))

    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')

# Build a breed-distrution chart from the filtered table records.
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
     # Use the filtered data currently displayed in the table
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)
        
    # Handle empty results
    if dff.empty or 'breed' not in dff.columns:
        return [
            html.Div(
                "No animal records match the selected filter.",
                style={'textAlign': 'center', 'padding': '30px'}
            )
        ]
    
    # Count animals by breed
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    # Create the interactive pie chart
    fig = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Animal Breed Distribution'
    )

    # Configure the pie chart appearance
    fig.update_traces(
        textposition='none',
        hovertemplate='<b>%{label}</b><extra></extra>'
    )

    fig.update_layout(
        margin=dict(l=20, r=20, t=60, b=20),
        legend_title_text='Breed',
        showlegend=False,
        height=450
    )

     # Return the pie chart to the dashboard
    return [
        dcc.Graph(
            figure=fig,
            style={'width': '100%', 'height': '450px'}
        )
    ]
    
# Highlight the selected data-table column.
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)

def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [
        {
            'if': {'column_id': column},
            'backgroundColor': '#D2F3FF'
        }
        for column in selected_columns
    ]



# Update the map when the filtered data or selected row changes.
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    
    # Use the currently filtered table data
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)
        
    # Default map location: Austin, Texas
    default_center = [30.75, -97.48]
    
    # Handle empty filtered results
    if dff.empty:
        return [
            html.Div(
                "No animal locations available for this filter.",
                style={'textAlign': 'center', 'padding': '30px'}
            )
        ]
    
    # Convert coordinates values to numbers and exclude invalid locations.
    dff['location_lat'] = pd.to_numeric(
        dff['location_lat'], errors='coerce'
    )
    dff['location_long'] = pd.to_numeric(
        dff['location_long'], errors='coerce'
    )
    
    dff = dff.dropna(subset=['location_lat', 'location_long'])
    
    # Handle records without valid coordinates
    if dff.empty:
        return [
            dl.Map(
                center=default_center,
                zoom=10,
                style={
                    'width': '100%',
                    'height': '450px'
                },
                children=[
                    dl.TileLayer()
                ]
            )
        ]
    
    # Determine the selected row
    row = 0
    
    if index is not None and len(index) > 0:
        selected_row = index[0]
        
        if selected_row < len(dff):
            row = selected_row
            
    # Retrieve animal information
    animal = dff.iloc[row]
    
    latitude = animal['location_lat']
    longitude = animal['location_long']
    
    animal_name = animal.get('name', 'Unknown')
    animal_breed = animal.get('breed', 'Unknown')
    
    # Create a map centered on teh selected animal
    return [
        dl.Map(
            style={'width': '100%', 'height': '450px'},
            center=[latitude, longitude],
            zoom=12,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(animal_breed),
                        dl.Popup([
                            html.H4("Animal Information"),
                            html.P(f"Name: {animal_name}"),
                            html.P(f"Breed: {animal_breed}")
                        ])
                    ]
                )
            ]
        )
    ]


# Start the interactive Dash dashboard
app.run_server() 

Dash app running on https://freedominitial-pigmentvoyage-3000.codio.io/proxy/8050/
